# Lorenz-63 data assimilation with weighted ensemble Jacobian guidance

This notebook demonstrates data assimilation for the Lorenz-63 system using a
conditional diffusion forecast model and observation-guided reverse diffusion.

## Workflow

1. **Burn in the Lorenz-63 truth model** and build an attractor-based pool for ensemble initialization.
2. **Load the trained conditional diffusion model** and normalization statistics.
3. **Define the observation operator and guided EDM sampler.**
4. **Run baselines**: particle filtering with the true model, particle filtering with the diffusion forecast model, and a diffusion-model free run.
5. **Tune and run the proposed DA method** using a nested search over the diagonal likelihood factor \(\Lambda_L\) and diagonal Jacobian factor \(\alpha_F\).
6. **Save the numerical results** for later MATLAB visualization.
7. **Run a lightweight Python diagnostic** of ensemble RMSE.

The Lorenz-63 defaults are
\[
\sigma=10,\qquad \rho=28,\qquad \beta=8/3,\qquad \Delta t=0.01.
\]

## Weighted ensemble Jacobian approximation

At each reverse-diffusion stage, let
\[
U_t^{(e)}=c_{\mathrm{in}}X_t^{(e)},\qquad
\bar U_t=\frac{1}{N_e}\sum_{e=1}^{N_e}U_t^{(e)},
\]
and recover the corresponding network quantity \(F_\theta(U_t^{(e)},\sigma_t)\)
from the denoiser evaluations already required by the sampler. Define
\[
\bar F_t=\frac{1}{N_e}\sum_{e=1}^{N_e}F_\theta(U_t^{(e)},\sigma_t).
\]
A single full \(3\times3\) Jacobian is estimated from
\[
F_\theta(U_t^{(e)},\sigma_t)-\bar F_t
\approx
J_F(\bar U_t)\left(U_t^{(e)}-\bar U_t\right)
\]
using Gaussian weighted least squares. Ensemble members closer to \(\bar U_t\)
receive larger weights. The adaptive bandwidth is
\[
h_t=\beta_h\,\operatorname{median}_e\|U_t^{(e)}-\bar U_t\|_2,
\]
where \(\beta_h\) is controlled by `JF_WEIGHT_BANDWIDTH_FACTOR`.

## Before running

- Set `NETWORK_PKL` to the trained conditional diffusion checkpoint.
- Check the Lorenz-63 configuration and ensemble/DA settings in the user-configuration cell.
- Edit `LAMBDA_L_SETS` and `ALPHA_F_SETS` if a different tuning grid is desired.
- In each saving section, choose your own `folder1` and `filename_save`.
- Run the notebook from top to bottom so that all methods use the same truth trajectory and random-seed conventions.


In [ ]:
# %pip install pandas

In [ ]:
import os
import pickle
import time

import dnnlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from project_utils.normalization import (
    denormalize_tensor,
    load_norm_stats,
    normalize_tensor,
    resolve_norm_stats_path,
)
from project_utils.paths import PROJECT_ROOT, CKPT_DIR, dataset_trajectory_dir
from project_utils.yaml_simple import load_simple_yaml
from turbulence.lorenz63 import Lorenz63Params, Lorenz63Solver

### Utilities

**Shape convention:** `C=1`, sequence length `3`, stored as `(1, 3, 1)` for `(x, y, z)`.

In [ ]:
class StackedRandomGenerator:
    def __init__(self, device, seeds):
        self.generators = [
            torch.Generator(device).manual_seed(int(s) % (1 << 32)) for s in seeds
        ]

    def randn(self, size, **kwargs):
        if len(size) < 1 or len(self.generators) != size[0]:
            raise ValueError(
                f"size[0]={size[0] if size else None} must match n_generators={len(self.generators)}"
            )
        return torch.stack(
            [torch.randn(size[1:], generator=g, **kwargs) for g in self.generators]
        )

    def randn_like(self, input):
        return self.randn(
            input.shape, dtype=input.dtype, layout=input.layout, device=input.device
        )


def frame_tensor(x: torch.Tensor) -> torch.Tensor:
    return x.reshape(1, STATE_DIM, 1)


def make_initial_condition(x0, *, device, dtype: torch.dtype) -> torch.Tensor:
    return torch.tensor(list(x0), device=device, dtype=dtype)


def build_condition_from_history(frames_phys, history, norm_stats):
    if isinstance(frames_phys, torch.Tensor):
        recent = frames_phys[-history:]
        normed = [normalize_tensor(recent[i], norm_stats) for i in range(recent.shape[0])]
        block = torch.stack(normed, dim=0)
    else:
        recent = frames_phys[-history:]
        normed = [normalize_tensor(f, norm_stats) for f in recent]
        block = torch.stack(normed, dim=0)
    C, H, W = block.shape[1], block.shape[2], block.shape[3]
    return block.reshape(history * C, H, W).unsqueeze(0)

def build_condition_from_history_batch(histories, history, norm_stats):
    return torch.cat(
        [build_condition_from_history(h, history, norm_stats) for h in histories],
        dim=0,
    )

def perturb_state(frame, std, seed):
    g = torch.Generator().manual_seed(int(seed) % (1 << 32))
    return frame + std * torch.randn(frame.shape, generator=g, dtype=frame.dtype, device=frame.device)


def ensemble_last_frames(histories):
    return torch.stack([h[-1] for h in histories], dim=0)


def ensemble_mean_rmse(histories, x_truth):
    ens = ensemble_last_frames(histories)
    mean = ens.mean(dim=0)
    spread = ens.std(dim=0, correction=0).mean().item() if ens.shape[0] > 1 else 0.0
    rmse = torch.sqrt(torch.mean((mean - x_truth) ** 2)).item()
    return rmse, spread


def ensemble_mean_rmse_at_idx(histories, x_truth, flat_idx):
    """RMSE of ensemble mean vs truth on selected flat indices into (C,H,W)."""
    if flat_idx is None or flat_idx.numel() == 0:
        return float("nan")
    ens = ensemble_last_frames(histories)
    mean = ens.mean(dim=0).reshape(-1)
    truth = x_truth.reshape(-1)
    idx = flat_idx.cpu()
    return torch.sqrt(torch.mean((mean[idx] - truth[idx]) ** 2)).item()


def to_seq(traj: torch.Tensor) -> torch.Tensor:
    return traj[:, 0, :, 0]

In [ ]:
# --- User configuration ---
L63_CONFIG = os.path.join(PROJECT_ROOT, "configs/turbulence/lorenz63_default.yaml")
DATA_ROOT = "LORENZ63"
TRAJECTORY_SUBDIR = "full_path"

# Path to trained checkpoint (network-snapshot-*.pkl from train_cond.py)
NETWORK_PKL = os.path.join(
    CKPT_DIR,
    "l63.pkl",
)


DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
SEED = 0
SEED_DM = 0

STATE_DIM = 3
VAR_LABELS = ["x", "y", "z"]
X0 = [0.0, 1.0, 0.0]

# X0 = [1.0, 0.0, 0.0]
# Total pre-DA trajectory length. The final state is used as the truth fork state.
T_BURN_IN = 8000

# T_BURN_IN = 9900
# T_BURN_IN = 10300


# Build initial ensemble members from states already on the L63 attractor.
# The first ATTRACTOR_DISCARD output steps are treated as transient and excluded.
ATTRACTOR_DISCARD = 1000
ATTRACTOR_THIN = 5      # 1 = no thinning; e.g. 10 keeps every 10th post-transient state
INIT_POOL_SEED = SEED + 4242
# INIT_POOL_SEED = SEED + 3242
N_DA_CYCLES = 40

# Match trained DM stride. DART classic uses 0.06 (6 model steps); regenerate data if you change this.
OUTPUT_INTERVAL = 0.01

HISTORY = 1

NE = 800
# No Gaussian perturbation is used for the new attractor-based initialization.
# Kept here only for easy comparison with the old initialization.
ENSEMBLE_PERTURB_STD = 0.5

device = torch.device(DEVICE)
trajectory_dir = dataset_trajectory_dir(DATA_ROOT, TRAJECTORY_SUBDIR)
norm_stats_path = resolve_norm_stats_path(trajectory_dir, None)

## Phase 1: True L63 burn-in

In [ ]:
cfg = load_simple_yaml(L63_CONFIG)
l63_cfg = dict(cfg.get("lorenz63", {}))
l63_cfg["device"] = str(device)

params = Lorenz63Params(**l63_cfg)
params.validate()
timesteps_per_output = max(1, int(round(OUTPUT_INTERVAL / params.dt)))
dtype = torch.float32 if params.precision == "single" else torch.float64

x0 = make_initial_condition(X0, device=device, dtype=dtype)
solver = Lorenz63Solver(x0, **params.to_core_kwargs())

burn_frames = [frame_tensor(solver.fields()["x"].detach().cpu())]
for _ in tqdm(range(T_BURN_IN), desc="L63 burn-in"):
    solver.step(timesteps_per_output)
    burn_frames.append(frame_tensor(solver.fields()["x"].detach().cpu()))

state_after_burn = {k: v.clone() for k, v in solver.get_state().items()}
path_burn = torch.stack(burn_frames, dim=0)
print("Burn-in frames:", tuple(path_burn.shape))
print("Last (x,y,z):", path_burn[-1, 0, :, 0].tolist())

# ------------------------------------------------------------------
# Attractor-based initial ensemble pool
# ------------------------------------------------------------------
# Exclude the transient and also exclude the final burn-in state itself,
# because path_burn[-1] is the common truth state at the start of DA.
candidate_state_indices = torch.arange(
    ATTRACTOR_DISCARD, path_burn.shape[0] - 1, ATTRACTOR_THIN, dtype=torch.long
)

if candidate_state_indices.numel() < NE:
    raise ValueError(
        f"Only {candidate_state_indices.numel()} post-transient states are available, "
        f"but NE={NE}. Increase T_BURN_IN or reduce ATTRACTOR_DISCARD."
    )

init_gen = torch.Generator(device="cpu")
init_gen.manual_seed(INIT_POOL_SEED)
init_perm = torch.randperm(candidate_state_indices.numel(), generator=init_gen)
init_state_indices = candidate_state_indices[init_perm]

print("Attractor candidate states:", candidate_state_indices.numel(), f"(thin={ATTRACTOR_THIN})")
print("First selected state indices:", init_state_indices[:10].tolist())


# def make_attractor_initialized_history(path, state_idx, history_len=1):
#     """
#     Keep the existing history-list structure, but replace the final
#     `history_len` entries by a dynamically consistent segment ending
#     at the selected attractor state.

#     For HISTORY=1, this simply replaces the final state.
#     """
#     state_idx = int(state_idx)
#     history_len = int(history_len)
#     start_idx = state_idx - history_len + 1
#     if start_idx < ATTRACTOR_DISCARD:
#         raise ValueError("Selected attractor state does not have enough post-transient history.")

#     hist = [path[i].clone() for i in range(path.shape[0])]
#     hist[-history_len:] = [path[j].clone() for j in range(start_idx, state_idx + 1)]
#     return hist
def make_attractor_initialized_history(path, state_idx, history_len=1):
    state_idx = int(state_idx)
    history_len = int(history_len)

    start_idx = state_idx - history_len + 1

    if start_idx < ATTRACTOR_DISCARD:
        raise ValueError(
            "Selected attractor state does not have enough post-transient history."
        )

    return [
        path[j].clone()
        for j in range(start_idx, state_idx + 1)
    ]


## Phase 2: Load trained conditional diffusion model + norm stats

In [ ]:
if not os.path.isfile(NETWORK_PKL):
    raise FileNotFoundError(
        f"Set NETWORK_PKL. Not found: {NETWORK_PKL}\n"
        "Train with: torchrun scripts/train/train_cond.py runs=lorenz63_path_next"
    )

norm_stats = load_norm_stats(norm_stats_path)
with dnnlib.util.open_url(NETWORK_PKL, verbose=True) as f:
    net = pickle.load(f)["ema"].to(device).eval()

n_target = 1
n_cond = HISTORY * n_target
assert net.img_input_channels == n_target
assert net.img_cond_channels == n_cond
assert net.img_res_h == STATE_DIM and net.img_res_w == 1

print(f"Loaded {NETWORK_PKL}")
print(f"Model: {net.model_name}")

### Observation operator (direct state)

DART L63 benchmark observes **all three** variables. Likelihood guidance is applied inside
`edm_sampler_obs` when `obs["y_obs"]` is set.

In [ ]:
def random_obs_flat_indices(state_dim, obs_count, obs_channels, seed=0, device="cpu"):
    rng = np.random.default_rng(seed)
    all_idx = []
    for c in obs_channels:
        base = int(c) * int(state_dim)
        all_idx.extend(range(base, base + int(state_dim)))
    flat_idx = rng.choice(np.array(all_idx, dtype=np.int64), size=obs_count, replace=False)
    return torch.tensor(flat_idx, dtype=torch.long, device=device)


def obs_mask_vector(flat_idx, total_size, device=None, dtype=torch.float32):
    dev = device if device is not None else flat_idx.device
    m = torch.zeros(total_size, device=dev, dtype=dtype)
    m[flat_idx.to(dev)] = 1.0
    return m


def observe_state(x_state, flat_idx, obs_sigma, generator=None):
    flat = x_state.reshape(-1)
    idx = flat_idx.to(flat.device)

    y = flat[idx].to(torch.float32)

    if obs_sigma > 0:
        y = y + obs_sigma * torch.randn(
            y.shape,
            device=y.device,
            generator=generator
        )

    return y


def observation_rmse(x_phys, y_obs, flat_idx):
    flat = x_phys.reshape(-1)
    idx = flat_idx.cpu()
    return torch.sqrt(torch.mean((flat[idx] - y_obs.reshape(-1).cpu()) ** 2)).item()

### EDM sampling (`edm_sampler_obs`)

Non-gradient observation guidance (no backprop through denoiser).

In [ ]:
def phys_scale_at_flat_idx(flat_idx, norm_stats, seq_len, n_channels, device, dtype=torch.float64):
    flat_idx = flat_idx.to(device)
    block = int(seq_len)
    channels = flat_idx // block
    if (channels < 0).any() or (channels >= n_channels).any():
        raise ValueError(
            f"flat_idx out of range for (C={n_channels}, K={seq_len}) channel-major layout"
        )
    if norm_stats.get("scope", "global") == "global":
        s = 0.5 * float(norm_stats["vmax"] - norm_stats["vmin"])
        return torch.full((flat_idx.numel(),), s, device=device, dtype=dtype)
    ch_scale = 0.5 * (
        norm_stats["vmax_per_channel"] - norm_stats["vmin_per_channel"]
    ).to(device=device, dtype=dtype)
    return ch_scale[channels]


def check_obs_guidance_setup(norm_stats, flat_idx, n_channels, seq_len, device="cpu"):
    if norm_stats.get("scope", "global") == "global":
        lo = float(norm_stats["vmin"])
        hi = float(norm_stats["vmax"])
        x = torch.rand(n_channels, seq_len, 1, device=device) * (hi - lo) + lo
    else:
        vmin_c = norm_stats["vmin_per_channel"].to(device)
        vmax_c = norm_stats["vmax_per_channel"].to(device)
        x = torch.rand(n_channels, seq_len, 1, device=device)
        for c in range(n_channels):
            x[c] = x[c] * float(vmax_c[c] - vmin_c[c]) + float(vmin_c[c])
    err = (denormalize_tensor(normalize_tensor(x, norm_stats), norm_stats) - x).abs().max()
    if float(err) > 1e-4:
        raise RuntimeError(f"norm/denorm round-trip failed: max err {float(err):.2e}")
    print(f"obs guidance checks OK (round-trip err={float(err):.2e})")


def _edm_sigma_data(net, fallback=0.5):
    """Read the EDM preconditioning sigma_data from the network when available."""
    value = getattr(net, "sigma_data", fallback)
    try:
        return float(torch.as_tensor(value).detach().cpu().reshape(-1)[0].item())
    except Exception:
        return float(value)


def _c_skip(t_stage, sigma_data, device, dtype):
    """EDM skip coefficient: sigma_data^2 / (sigma^2 + sigma_data^2)."""
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return sd.pow(2) / (sigma.pow(2) + sd.pow(2))


def _c_out_c_in(t_stage, sigma_data, device, dtype):
    """Product c_out*c_in = sigma*sigma_data / (sigma^2 + sigma_data^2)."""
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return sigma * sd / (sigma.pow(2) + sd.pow(2))


def _c_in(t_stage, sigma_data, device, dtype):
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return 1.0 / torch.sqrt(sigma.pow(2) + sd.pow(2))


def _c_out(t_stage, sigma_data, device, dtype):
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return sigma * sd / torch.sqrt(sigma.pow(2) + sd.pow(2))


def _estimate_weighted_fbar_jf(x_noisy, x0_denoised, t_stage, sigma_data, bandwidth_factor=1.0, ridge_rel=1e-6, ridge_abs=1e-10, eps=1e-12):
    """Estimate one full J_F using weighted ensemble least squares around Ubar.

    Let U = c_in X, Ubar = mean(U), and Fbar = mean(F_theta(U_e)).
    The regression is
        F_theta(U_e) - Fbar ~= J_F(Ubar) (U_e - Ubar).

    Gaussian weights emphasize ensemble members closer to Ubar:
        w_e = exp(-||U_e-Ubar||^2 / (2 h^2)),
        h = bandwidth_factor * median_e ||U_e-Ubar||.

    No additional neural-network evaluation is required: F_theta(U_e) is
    recovered from the denoiser outputs already computed by the sampler.
    """
    device, dtype = x_noisy.device, x_noisy.dtype
    cskip = _c_skip(t_stage, sigma_data, device, dtype)
    cin = _c_in(t_stage, sigma_data, device, dtype)
    cout = _c_out(t_stage, sigma_data, device, dtype)

    u = cin * x_noisy
    f = (x0_denoised - cskip * x_noisy) / torch.clamp(cout, min=eps)

    uf = u.reshape(u.shape[0], -1)
    ff = f.reshape(f.shape[0], -1)

    u_bar = uf.mean(dim=0, keepdim=True)
    f_bar = ff.mean(dim=0, keepdim=True)

    du = uf - u_bar
    df = ff - f_bar

    distances = torch.linalg.vector_norm(du, ord=2, dim=1)
    positive_distances = distances[distances > eps]
    if positive_distances.numel() > 0:
        median_distance = torch.median(positive_distances)
    else:
        median_distance = torch.as_tensor(1.0, device=device, dtype=dtype)

    h = torch.as_tensor(float(bandwidth_factor), device=device, dtype=dtype) * median_distance
    h = torch.clamp(h, min=eps)

    weights = torch.exp(-0.5 * (distances / h).pow(2))
    weights = weights / torch.clamp(weights.mean(), min=eps)
    sqrt_w = torch.sqrt(torch.clamp(weights, min=0.0)).unsqueeze(1)

    du_w = sqrt_w * du
    df_w = sqrt_w * df

    gram = du_w.transpose(0, 1) @ du_w
    rhs = du_w.transpose(0, 1) @ df_w

    d = gram.shape[0]
    gram_scale = torch.trace(gram).abs() / max(d, 1)
    ridge = torch.as_tensor(float(ridge_abs), device=device, dtype=dtype) + torch.as_tensor(float(ridge_rel), device=device, dtype=dtype) * gram_scale
    eye = torch.eye(d, device=device, dtype=dtype)

    jf_t = torch.linalg.solve(gram + ridge * eye, rhs)
    jf = jf_t.transpose(0, 1)
    return jf


def _apply_jf_transpose(jf, g):
    """Apply J_F^T to each ensemble guidance vector without forming per-member Jacobians."""
    g_flat = g.reshape(g.shape[0], -1)
    # For row vectors: (J_F^T g_col)^T = g_row @ J_F.
    out_flat = g_flat @ jf
    return out_flat.reshape_as(g)


def edm_sampler_obs(
    net, latents, conditions, obs=None, class_labels=None, randn_like=torch.randn_like,
    num_steps=18, sigma_min=0.002, sigma_max=80, rho=7,
    S_churn=0, S_min=0, S_max=float("inf"), S_noise=1
):
    # Weighted Fbar-centered J_F regression with componentwise asymmetric two-stage Heun guidance.
    # Stage 1 uses the intermediate diagonal multiplier; stage 2 uses the baseline likelihood scale.
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    use_obs = obs is not None and obs.get("y_obs") is not None

    if use_obs:
        device = latents.device
        flat_idx = obs["flat_idx"].to(device)
        obs_sigma = float(obs["obs_sigma"])
        sigma_data = float(obs.get("sigma_data", 1.0))
        use_denom = bool(obs.get("use_denom", False))
        norm_stats = obs["norm_stats"]

        # IMPORTANT: this is the sigma_data used by EDM preconditioning,
        # not the optional sigma_data used in the observation denominator.
        edm_sigma_data = _edm_sigma_data(
            net, fallback=float(obs.get("edm_sigma_data_fallback", 0.5))
        )

        residual_soften_alpha = float(obs.get("residual_soften_alpha", 1.0))
        jf_weight_bandwidth_factor = float(obs.get("jf_weight_bandwidth_factor", 1.0))
        jf_ridge_rel = float(obs.get("jf_ridge_rel", 1e-6))
        jf_ridge_abs = float(obs.get("jf_ridge_abs", 1e-10))
        alpha_f_x = float(obs.get("alpha_f_x", 1.0))
        alpha_f_y = float(obs.get("alpha_f_y", 1.0))
        alpha_f_z = float(obs.get("alpha_f_z", 1.0))
        likelihood_scale_x = float(obs.get("likelihood_scale_x", 1.0))
        likelihood_scale_y = float(obs.get("likelihood_scale_y", 1.0))
        likelihood_scale_z = float(obs.get("likelihood_scale_z", 1.0))
        intermediate_guidance_multiplier_x = float(
            obs.get("intermediate_guidance_multiplier_x", 1.0)
        )
        intermediate_guidance_multiplier_y = float(
            obs.get("intermediate_guidance_multiplier_y", 1.0)
        )
        intermediate_guidance_multiplier_z = float(
            obs.get("intermediate_guidance_multiplier_z", 1.0)
        )
        residual_soften_c = max(
            residual_soften_alpha * np.sqrt(float(flat_idx.numel())) * obs_sigma,
            1e-12,
        )

        phi = obs.get("phi", lambda u: u)
        phi_prime = obs.get("phi_prime", lambda u: torch.ones_like(u))

        y_obs = obs["y_obs"].to(device, torch.float64)
        if y_obs.dim() == 1:
            y_obs = y_obs.unsqueeze(0)

    step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)

    t_steps = (
        sigma_max ** (1 / rho)
        + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
    ) ** rho

    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])])

    # Inverse-relative |dt| compensation:
    # adaptive_dt_factor = mean(|dt|)/|dt_i|.
    all_abs_dt = torch.abs(t_steps[1:] - t_steps[:-1])
    mean_abs_dt = all_abs_dt.mean().clamp_min(1e-12)

    x_next = latents.to(torch.float64) * t_steps[0]
    neg_log_likelihoods = [] if use_obs else None

    def likelihood_grad_from_denoised(x0_norm, sigma_for_denom):
        """Return grad log p(y|x0), residual, and per-member NLL at this Heun stage."""
        x0_leaf = x0_norm.detach().requires_grad_(True)
        x0_flat = x0_leaf.reshape(x0_leaf.shape[0], -1)

        gx = phi(x0_flat[:, flat_idx])
        resid = gx - y_obs

        if use_denom:
            sigma_val = float(torch.as_tensor(sigma_for_denom).reshape(-1)[0].item())
            r2 = sigma_val**2 * sigma_data**2 / (sigma_val**2 + sigma_data**2)
            pp = phi_prime(x0_flat[:, flat_idx].detach())
            denom = obs_sigma**2 + r2 * pp.pow(2)
        else:
            denom = 1.0

        nll = 0.5 * (resid.pow(2) / denom).sum(dim=1)
        log_p = -nll.sum()
        guidance = torch.autograd.grad(log_p, x0_leaf)[0]
        return guidance.detach(), resid.detach(), nll.detach()

    # Componentwise likelihood strengths with a diagonal stage-1 multiplier.
    # alpha_F controls componentwise scaling of the recovered J_F action.
    if use_obs:
        alpha_f_component_scale = torch.tensor(
            [alpha_f_x, alpha_f_y, alpha_f_z],
            device=latents.device,
            dtype=latents.dtype,
        ).reshape(1, 1, 3, 1)

        # Use the intermediate multiplier at Heun stage 1 (predictor),
        # while stage 2 uses the baseline componentwise likelihood scale.
        likelihood_component_scale_stage2 = torch.tensor(
            [likelihood_scale_x, likelihood_scale_y, likelihood_scale_z],
            device=latents.device,
            dtype=latents.dtype,
        ).reshape(1, 1, 3, 1)

        intermediate_guidance_multiplier = torch.tensor(
            [
                intermediate_guidance_multiplier_x,
                intermediate_guidance_multiplier_y,
                intermediate_guidance_multiplier_z,
            ],
            device=latents.device,
            dtype=latents.dtype,
        ).reshape(1, 1, 3, 1)

        likelihood_component_scale_stage1 = (
            intermediate_guidance_multiplier * likelihood_component_scale_stage2
        )

    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])):
        x_cur = x_next

        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat**2 - t_cur**2).sqrt() * S_noise * randn_like(x_cur)
        dt = t_next - t_hat  # negative in reverse diffusion
        abs_dt = torch.abs(dt).clamp_min(1e-12)
        adaptive_dt_factor = mean_abs_dt / abs_dt

        with torch.no_grad():
            x0_cur = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)
        d_cur = (x_hat - x0_cur) / t_hat

        g_cur = None
        nll_cur = None
        scale_cur = None

        if use_obs:
            g_cur, resid_cur, nll_cur = likelihood_grad_from_denoised(x0_cur, t_hat)
            resid_norm_cur = torch.linalg.vector_norm(resid_cur, ord=2, dim=1)
            soft_resid_norm_cur = torch.sqrt(
                resid_norm_cur.pow(2) + residual_soften_c**2
            )
            cskip_cur = _c_skip(t_hat, edm_sigma_data, g_cur.device, g_cur.dtype)
            coutcin_cur = _c_out_c_in(t_hat, edm_sigma_data, g_cur.device, g_cur.dtype)
            jf_cur = _estimate_weighted_fbar_jf(x_hat, x0_cur, t_hat, edm_sigma_data, bandwidth_factor=jf_weight_bandwidth_factor, ridge_rel=jf_ridge_rel, ridge_abs=jf_ridge_abs).to(device=g_cur.device, dtype=g_cur.dtype)
            jf_t_g_cur = _apply_jf_transpose(jf_cur, g_cur)
            jac_action_cur = (
                cskip_cur * g_cur
                + alpha_f_component_scale * coutcin_cur * jf_t_g_cur
            )
            rshape_cur = [soft_resid_norm_cur.shape[0]] + [1] * (g_cur.ndim - 1)
            scale_cur = (
                adaptive_dt_factor
                / soft_resid_norm_cur.reshape(rshape_cur)
            )

        # Predictor: stage-1 likelihood guidance uses the intermediate multiplier.
        x_euler = x_hat + dt * d_cur
        if use_obs:
            x_euler = (
                x_euler
                - dt
                * t_hat
                * scale_cur
                * likelihood_component_scale_stage1
                * jac_action_cur
            )

        if i < num_steps - 1:
            with torch.no_grad():
                x0_prime = net(x_euler, t_next, conditions, class_labels).to(torch.float64)
            d_prime = (x_euler - x0_prime) / t_next

            g_prime = None
            nll_prime = None
            scale_prime = None

            if use_obs:
                g_prime, resid_prime, nll_prime = likelihood_grad_from_denoised(x0_prime, t_next)
                resid_norm_prime = torch.linalg.vector_norm(resid_prime, ord=2, dim=1)
                soft_resid_norm_prime = torch.sqrt(
                    resid_norm_prime.pow(2) + residual_soften_c**2
                )
                cskip_prime = _c_skip(t_next, edm_sigma_data, g_prime.device, g_prime.dtype)
                coutcin_prime = _c_out_c_in(t_next, edm_sigma_data, g_prime.device, g_prime.dtype)
                jf_prime = _estimate_weighted_fbar_jf(x_euler, x0_prime, t_next, edm_sigma_data, bandwidth_factor=jf_weight_bandwidth_factor, ridge_rel=jf_ridge_rel, ridge_abs=jf_ridge_abs).to(device=g_prime.device, dtype=g_prime.dtype)
                jf_t_g_prime = _apply_jf_transpose(jf_prime, g_prime)
                jac_action_prime = (
                    cskip_prime * g_prime
                    + alpha_f_component_scale * coutcin_prime * jf_t_g_prime
                )
                rshape_prime = [soft_resid_norm_prime.shape[0]] + [1] * (g_prime.ndim - 1)
                scale_prime = (
                    adaptive_dt_factor
                    / soft_resid_norm_prime.reshape(rshape_prime)
                )

            # Heun corrector: stage 1 uses the intermediate multiplier and stage 2 uses the baseline scale.
            x_next = x_hat + dt * 0.5 * (d_cur + d_prime)
            if use_obs:
                x_next = x_next - dt * 0.5 * (
                    t_hat
                    * scale_cur
                    * likelihood_component_scale_stage1
                    * jac_action_cur
                    + t_next
                    * scale_prime
                    * likelihood_component_scale_stage2
                    * jac_action_prime
                )
                neg_log_likelihoods.append(float(0.5 * (nll_cur.mean() + nll_prime.mean()).cpu()))
        else:
            x_next = x_euler
            if use_obs:
                neg_log_likelihoods.append(float(nll_cur.mean().cpu()))

    if use_obs:
        return x_next, neg_log_likelihoods
    return x_next


## Phase 3: Observation-guided sampler and baseline utilities

The sampler evaluates the likelihood correction at both Heun stages. Likelihood
guidance uses residual-norm scaling, and no additional diffusion timestep factor
is applied to the likelihood correction.

The following cells also define particle-filter utilities used by the baseline
experiments.


In [ ]:
def systematic_resample(weights):
    """
    Systematic resampling.

    weights: torch tensor of shape (Np,), normalized and nonnegative.
    returns: torch long tensor of shape (Np,)
    """
    device = weights.device
    Np = weights.numel()

    positions = (torch.arange(Np, device=device, dtype=weights.dtype) + torch.rand(1, device=device)) / Np
    cumsum = torch.cumsum(weights, dim=0)

    idx = torch.searchsorted(cumsum, positions)
    idx = torch.clamp(idx, max=Np - 1)

    return idx.long()

In [ ]:
def clone_history_item(hist_item):
    """
    hist_item is usually a list/deque of tensors.
    This creates an independent copy.
    """
    return [x.clone() for x in hist_item]


def resample_histories(hist_particles, resample_idx):
    """
    Resample the entire history for each particle.
    This is important because the diffusion model conditions on history.
    """
    idx_cpu = resample_idx.detach().cpu().numpy()

    new_hist = []
    for j in idx_cpu:
        new_hist.append(clone_history_item(hist_particles[int(j)]))

    return new_hist


def compute_log_weights_l63(x_particles_norm, y_obs, obs_flat_idx, obs_sigma):
    """
    Compute log likelihood weights for L63.

    x_particles_norm: Np x 1 x 3 x 1 or Np x 3
    y_obs: 1 x obs_count or obs_count
    obs_flat_idx: tensor of observed component indices
    obs_sigma: observation noise in normalized space
    """

    if x_particles_norm.dim() == 4:
        # Np x 1 x 3 x 1 -> Np x 3
        x_flat = x_particles_norm.reshape(x_particles_norm.shape[0], -1)
    elif x_particles_norm.dim() == 2:
        x_flat = x_particles_norm
    else:
        raise ValueError(f"Unexpected x_particles_norm shape: {x_particles_norm.shape}")

    y_obs = y_obs.to(x_flat.device, x_flat.dtype)
    if y_obs.dim() == 1:
        y_obs = y_obs.unsqueeze(0)

    obs_flat_idx = obs_flat_idx.to(x_flat.device)

    pred_obs = x_flat[:, obs_flat_idx]       # Np x obs_count
    resid = pred_obs - y_obs                 # Np x obs_count

    logw = -0.5 * torch.sum(resid**2, dim=1) / (obs_sigma**2)

    return logw


def normalize_log_weights(logw):
    """
    Stable normalization of log weights.
    """
    logw = logw - torch.max(logw)
    w = torch.exp(logw)
    w_sum = torch.sum(w)

    if not torch.isfinite(w_sum) or w_sum <= 0:
        # fallback to uniform weights
        w = torch.ones_like(w) / w.numel()
    else:
        w = w / w_sum

    return w


def effective_sample_size(weights):
    """
    ESS = 1 / sum w_i^2.
    """
    return 1.0 / torch.sum(weights**2)

In [ ]:
NUM_STEPS = 60

In [ ]:
n_burn = path_burn.shape[0]

In [ ]:
burn_norm = [normalize_tensor(path_burn[i], norm_stats) for i in range(n_burn)]

In [ ]:
obs_sigma = 0.1

## Phase 4A: Baseline experiments

### Particle filter with the true Lorenz-63 forecast model

This baseline propagates particles with the numerical Lorenz-63 solver, applies
the same observation sequence used by the DA experiment, and performs
likelihood weighting/resampling.


In [ ]:
core_kw = params.to_core_kwargs()
truth = Lorenz63Solver(path_burn[-1][0, :, 0].to(device=device, dtype=dtype), **core_kw)
truth.set_state(state_after_burn)

In [ ]:
def lorenz63_rhs(x, sigma=10.0, rho=28.0, beta=8.0/3.0):
    """
    x: tensor with shape (..., 3)
    returns dx/dt with shape (..., 3)
    """
    dx = torch.empty_like(x)

    dx[..., 0] = sigma * (x[..., 1] - x[..., 0])
    dx[..., 1] = x[..., 0] * (rho - x[..., 2]) - x[..., 1]
    dx[..., 2] = x[..., 0] * x[..., 1] - beta * x[..., 2]

    return dx


def rk4_lorenz63_step(x, dt=0.01, sigma=10.0, rho=28.0, beta=8.0/3.0):
    """
    One RK4 step for Lorenz-63.

    x: tensor with shape (..., 3)
    """
    k1 = lorenz63_rhs(x, sigma=sigma, rho=rho, beta=beta)
    k2 = lorenz63_rhs(x + 0.5 * dt * k1, sigma=sigma, rho=rho, beta=beta)
    k3 = lorenz63_rhs(x + 0.5 * dt * k2, sigma=sigma, rho=rho, beta=beta)
    k4 = lorenz63_rhs(x + dt * k3, sigma=sigma, rho=rho, beta=beta)

    return x + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


def propagate_lorenz63(x, n_steps, dt=0.01, sigma=10.0, rho=28.0, beta=8.0/3.0):
    """
    Propagate Lorenz-63 for n_steps RK4 steps.

    x: tensor with shape (NPF, 3) or (..., 3)
    """
    for _ in range(n_steps):
        x = rk4_lorenz63_step(x, dt=dt, sigma=sigma, rho=rho, beta=beta)
    return x

In [ ]:
hist_pf_true = []

In [ ]:
path_pf_analysis_true = []
path_pf_forecast_true = []

rmse_truth_vs_pf_true = []
spread_pf_true = []
# PF + true-model sample-impoverishment diagnostics.
ess_pf_true = []
unique_ancestors_pf_true = []
unique_states_pf_true = []
unique_states_pre_rejuv_pf_true = []

In [ ]:
# NPF = NE
NPF = 800

In [ ]:
if NPF > init_state_indices.numel():
    raise ValueError("NPF exceeds the number of available attractor states.")

for k in range(NPF):
    hf = make_attractor_initialized_history(path_burn, init_state_indices[k], HISTORY)
    hist_pf_true.append(hf)

# Pre-DA initial PF ensemble. Index 0 stores the pre-DA PF ensemble.
pf_true_initial_ens = torch.stack([hist_pf_true[k][-1] for k in range(NPF)], dim=0).clone()
path_pf_analysis_true = [pf_true_initial_ens.clone()]


In [ ]:
rnd_pf = StackedRandomGenerator(device, [SEED_DM + k for k in range(NPF)])

In [ ]:
all_idx = torch.arange(STATE_DIM, dtype=torch.long)

In [ ]:
obs_count = 3

In [ ]:
def make_random_l63_obs_idx(obs_count=2, state_dim=3, seed=None, device="cpu"):
    """
    Randomly choose obs_count state variables from {0,1,2}.
    For L63:
        0 -> x
        1 -> y
        2 -> z
    """
    if obs_count > state_dim:
        raise ValueError("obs_count cannot exceed state_dim")

    g = torch.Generator(device="cpu")
    if seed is not None:
        g.manual_seed(int(seed))

    idx = torch.randperm(state_dim, generator=g)[:obs_count]
    idx = torch.sort(idx)[0]
    return idx.to(device)

In [ ]:
obs_flat_idx = torch.tensor([0, 1, 2], dtype=torch.long, device=device)

In [ ]:
cycle_times = []

In [ ]:
# PF_TRUE_REJUV_STD = 0.0375
PF_TRUE_REJUV_STD = 0.02
PF_TRUE_REJUV_SEED = 61423

In [ ]:
pf_true_rejuv_gen = torch.Generator(device="cpu")
pf_true_rejuv_gen.manual_seed(int(PF_TRUE_REJUV_SEED))

pbar = tqdm(range(N_DA_CYCLES), desc="PF cycles", unit="cycle")

for cycle in pbar:
    t_cycle_start = time.perf_counter()

    # ------------------------------------------------------------
    # 1. Advance truth
    # ------------------------------------------------------------
    truth.step(timesteps_per_output)
    x_truth = frame_tensor(truth.fields()["x"].detach().cpu())      # physical
    x_truth_norm = normalize_tensor(x_truth, norm_stats)            # normalized

    obs_idx_cpu_pf = obs_flat_idx.detach().cpu()

    obs_noise_gen = torch.Generator(device=x_truth_norm.device)
    obs_noise_gen.manual_seed(SEED + 150000 + cycle)
    # Use normalized observation if EnSF likelihood is normalized.
    y_obs_pf = observe_state(x_truth_norm, obs_flat_idx, obs_sigma, generator=obs_noise_gen).to(device)

    # obs_values_pf.append(y_obs_pf.detach().cpu().reshape(-1))

    if not torch.isfinite(y_obs_pf).all():
        print(f"Stopping at cycle {cycle}: non-finite observations y_obs_pf")
        break

    # obs_rmse_pf.append(observation_rmse(x_truth_norm, y_obs_pf, obs_flat_idx))

    # ------------------------------------------------------------
    # 3. Forecast particles using true model
    # ------------------------------------------------------------
    x_pf_current = torch.stack([hist_pf_true[k][-1] for k in range(NPF)], dim=0)
    x_pf_current = x_pf_current.reshape(NPF, 3).to(device).to(torch.float32)
    
    # Propagate particles with true Lorenz-63 model
    with torch.no_grad():
        next_pf_forecast_flat = propagate_lorenz63(
            x_pf_current, n_steps=timesteps_per_output, dt=float(params.dt),
            sigma=10.0, rho=28.0, beta=8.0/3.0)
    
    # Convert back to your stored shape, probably NPF x 1 x 3 x 1
    next_pf_forecast = next_pf_forecast_flat.reshape(NPF, 1, 3, 1).cpu()

    if not torch.isfinite(next_pf_forecast).all():
        print(f"Stopping at cycle {cycle}: non-finite PF forecast state")
        break

    # Save forecast particles before resampling if desired
    path_pf_forecast_true.append(next_pf_forecast.clone())

    # ------------------------------------------------------------
    # 4. Compute likelihood weights in normalized space
    # ------------------------------------------------------------
    # sample_pf_norm has shape NPF x 1 x 3 x 1
    next_pf_forecast_norm = normalize_tensor(next_pf_forecast, norm_stats).to(device)

    logw = compute_log_weights_l63(
        next_pf_forecast_norm,
        y_obs_pf,
        obs_flat_idx,
        obs_sigma
    )
    
    weights = normalize_log_weights(logw)

    if not torch.isfinite(weights).all():
        print(f"Stopping at cycle {cycle}: non-finite PF weights")
        break

    ess = effective_sample_size(weights)
    ess_pf_true.append(float(ess.detach().cpu()))

    # ------------------------------------------------------------
    # 5. Append forecast state to histories, then resample histories
    # ------------------------------------------------------------
    # First append the forecast state x_k^f to each particle history.
    for k in range(NPF):
        hist_pf_true[k].append(next_pf_forecast[k])

    # Resample particles according to likelihood weights.
    resample_idx = systematic_resample(weights)

    n_unique_ancestors = int(torch.unique(resample_idx).numel())
    unique_ancestors_pf_true.append(n_unique_ancestors)
    
    # Resample the whole history, not just the last state.
    hist_pf_true = resample_histories(hist_pf_true, resample_idx)

    # Current posterior particles after resampling
    next_pf_analysis = torch.stack(
        [hist_pf_true[k][-1] for k in range(NPF)],
        dim=0
    )
    posterior_flat_pre = next_pf_analysis.reshape(NPF, -1)
    n_unique_states_pre = int(
        torch.unique(posterior_flat_pre, dim=0).shape[0]
    )
    unique_states_pre_rejuv_pf_true.append(n_unique_states_pre)

    if PF_TRUE_REJUV_STD > 0.0:
        rejuv_noise = PF_TRUE_REJUV_STD * torch.randn(
            next_pf_analysis.shape,
            generator=pf_true_rejuv_gen,
            dtype=next_pf_analysis.dtype,
            device="cpu",
        )

        # Zero empirical mean: add spread without directly moving the PF mean.
        rejuv_noise = rejuv_noise - rejuv_noise.mean(dim=0, keepdim=True)
        next_pf_analysis = next_pf_analysis + rejuv_noise

        # Future deterministic forecasts start from the rejuvenated posterior.
        for k in range(NPF):
            hist_pf_true[k][-1] = next_pf_analysis[k].clone()

    path_pf_analysis_true.append(next_pf_analysis.clone())

    posterior_flat = next_pf_analysis.reshape(NPF, -1)
    n_unique_states = int(torch.unique(posterior_flat, dim=0).shape[0])
    unique_states_pf_true.append(n_unique_states)
    
    # ------------------------------------------------------------
    # 6. Metrics
    # ------------------------------------------------------------
    rmse_pf, spr_pf = ensemble_mean_rmse(hist_pf_true, x_truth)

    if not all(np.isfinite([rmse_pf, spr_pf])):
        print(f"Stopping at cycle {cycle}: non-finite PF RMSE/spread metrics")
        break

    rmse_truth_vs_pf_true.append(rmse_pf)
    spread_pf_true.append(spr_pf)

    # ------------------------------------------------------------
    # 7. Progress bar
    # ------------------------------------------------------------
    cycle_times.append(time.perf_counter() - t_cycle_start)
    avg_step = sum(cycle_times) / len(cycle_times)
    remaining = avg_step * (N_DA_CYCLES - cycle - 1)

    pbar.set_postfix(
        rmse=f"{rmse_pf:.4f}",
        spr=f"{spr_pf:.4f}",
        ess=f"{float(ess):.1f}",
        uniq=f"{n_unique_states_pre}->{n_unique_states}/{NPF}",
        step=f"{avg_step:.2f}s",
        remaining=f"{remaining:.0f}s"
    )

# PF + true-model degeneracy / rejuvenation summary.
if unique_states_pf_true:
    print(
        "PF+true unique posterior states BEFORE rejuvenation: "
        f"final={unique_states_pre_rejuv_pf_true[-1]}/{NPF}, "
        f"minimum={min(unique_states_pre_rejuv_pf_true)}/{NPF}"
    )
    print(
        "PF+true unique posterior states AFTER rejuvenation: "
        f"final={unique_states_pf_true[-1]}/{NPF}, "
        f"minimum={min(unique_states_pf_true)}/{NPF}"
    )
    print(
        "PF+true unique resampled ancestors: "
        f"final={unique_ancestors_pf_true[-1]}/{NPF}, "
        f"minimum={min(unique_ancestors_pf_true)}/{NPF}"
    )

### Particle filter with the diffusion forecast model

This baseline replaces the true-model forecast step with samples from the
conditional diffusion forecast model while retaining the particle-filter
likelihood weighting/resampling step.


In [ ]:
hist_pf = []

In [ ]:
core_kw = params.to_core_kwargs()
truth = Lorenz63Solver(path_burn[-1][0, :, 0].to(device=device, dtype=dtype), **core_kw)
truth.set_state(state_after_burn)

In [ ]:
if NPF > init_state_indices.numel():
    raise ValueError("NPF exceeds the number of available attractor states.")

for k in range(NPF):
    hf = make_attractor_initialized_history(path_burn, init_state_indices[k], HISTORY)
    hist_pf.append(hf)

# Pre-DA initial PF ensemble. Index 0 stores the pre-DA PF ensemble.
pf_ai_initial_ens = torch.stack([hist_pf[k][-1] for k in range(NPF)], dim=0).clone()


In [ ]:
path_pf_analysis = [pf_ai_initial_ens.clone()]
path_pf_forecast = []

rmse_truth_vs_pf = []
spread_pf = []

ess_hist = []
obs_idx_hist_pf = []
unobs_idx_hist_pf = []
obs_values_pf = []
obs_rmse_pf = []

cycle_times = []

In [ ]:
pbar = tqdm(range(N_DA_CYCLES), desc="PF cycles", unit="cycle")

for cycle in pbar:
    t_cycle_start = time.perf_counter()

    # ------------------------------------------------------------
    # 1. Advance truth
    # ------------------------------------------------------------
    truth.step(timesteps_per_output)
    x_truth = frame_tensor(truth.fields()["x"].detach().cpu())      # physical
    x_truth_norm = normalize_tensor(x_truth, norm_stats)            # normalized

    obs_noise_gen = torch.Generator(device=x_truth_norm.device)
    obs_noise_gen.manual_seed(SEED + 150000 + cycle)
    # Use normalized observation if EnSF likelihood is normalized.
    y_obs_pf = observe_state(x_truth_norm, obs_flat_idx, obs_sigma, generator=obs_noise_gen).to(device)

    obs_values_pf.append(y_obs_pf.detach().cpu().reshape(-1))

    if not torch.isfinite(y_obs_pf).all():
        print(f"Stopping at cycle {cycle}: non-finite observations y_obs_pf")
        break

    obs_rmse_pf.append(observation_rmse(x_truth_norm, y_obs_pf, obs_flat_idx))

    # ------------------------------------------------------------
    # 3. Forecast particles using the same diffusion/free-run model
    # ------------------------------------------------------------
    cond_pf = build_condition_from_history_batch(hist_pf, HISTORY, norm_stats).to(device)

    if not torch.isfinite(cond_pf).all():
        print(f"Stopping at cycle {cycle}: non-finite PF conditions")
        break

    lat_pf = rnd_pf.randn([NPF, n_target, STATE_DIM, 1], device=device)

    with torch.no_grad():
        sample_pf_norm = edm_sampler_obs(
            net, lat_pf, cond_pf, obs=None,
            num_steps=NUM_STEPS
        ).to(torch.float32)

    if not torch.isfinite(sample_pf_norm).all():
        print(f"Stopping at cycle {cycle}: non-finite PF forecast sample")
        break

    # Physical forecast particles
    next_pf_forecast = denormalize_tensor(sample_pf_norm.cpu(), norm_stats)

    if not torch.isfinite(next_pf_forecast).all():
        print(f"Stopping at cycle {cycle}: non-finite PF forecast state")
        break

    # Save forecast particles before resampling if desired
    path_pf_forecast.append(next_pf_forecast.clone())

    # ------------------------------------------------------------
    # 4. Compute likelihood weights in normalized space
    # ------------------------------------------------------------
    # sample_pf_norm has shape NPF x 1 x 3 x 1
    logw = compute_log_weights_l63(
        sample_pf_norm,
        y_obs_pf,
        obs_flat_idx,
        obs_sigma
    )

    weights = normalize_log_weights(logw)

    if not torch.isfinite(weights).all():
        print(f"Stopping at cycle {cycle}: non-finite PF weights")
        break

    ess = effective_sample_size(weights)
    ess_hist.append(float(ess.detach().cpu()))

    # ------------------------------------------------------------
    # 5. Append forecast state to histories, then resample histories
    # ------------------------------------------------------------
    # First append the forecast state x_k^f to each particle history.
    for k in range(NPF):
        hist_pf[k].append(next_pf_forecast[k])

    # Resample particles according to likelihood weights.
    resample_idx = systematic_resample(weights)

    # Resample the whole history, not just the last state.
    hist_pf = resample_histories(hist_pf, resample_idx)

    # Current posterior particles after resampling
    next_pf_analysis = torch.stack(
        [hist_pf[k][-1] for k in range(NPF)],
        dim=0
    )

    path_pf_analysis.append(next_pf_analysis.clone())

    # ------------------------------------------------------------
    # 6. Metrics
    # ------------------------------------------------------------
    rmse_pf, spr_pf = ensemble_mean_rmse(hist_pf, x_truth)

    if not all(np.isfinite([rmse_pf, spr_pf])):
        print(f"Stopping at cycle {cycle}: non-finite PF RMSE/spread metrics")
        break

    rmse_truth_vs_pf.append(rmse_pf)
    spread_pf.append(spr_pf)

    # ------------------------------------------------------------
    # 7. Progress bar
    # ------------------------------------------------------------
    cycle_times.append(time.perf_counter() - t_cycle_start)
    avg_step = sum(cycle_times) / len(cycle_times)
    remaining = avg_step * (N_DA_CYCLES - cycle - 1)

    pbar.set_postfix(
        rmse=f"{rmse_pf:.4f}",
        spr=f"{spr_pf:.4f}",
        ess=f"{float(ess):.1f}",
        step=f"{avg_step:.2f}s",
        remaining=f"{remaining:.0f}s"
    )

In [ ]:
n_da_cycles_completed = len(rmse_truth_vs_pf)
print(
    f"Completed {n_da_cycles_completed}/{N_DA_CYCLES} DA cycles. "
    f"forecast {rmse_truth_vs_pf[-1] if rmse_truth_vs_pf else float('nan'):.4f}"
)
if cycle_times:
    print(f"Avg time per cycle: {sum(cycle_times) / len(cycle_times):.2f}s")

In [ ]:
truth_frames = [path_burn[i].clone() for i in range(path_burn.shape[0])]
truth_frames_norm = [burn_norm[i].clone() for i in range(path_burn.shape[0])]
truth_solver_log = Lorenz63Solver(path_burn[-1][0, :, 0].to(device=device, dtype=dtype), **core_kw)
truth_solver_log.set_state(state_after_burn)

In [ ]:
for _ in range(n_da_cycles_completed):
    truth_solver_log.step(timesteps_per_output)
    x_truth_log = frame_tensor(truth_solver_log.fields()["x"].detach().cpu())
    x_truth_log_norm = normalize_tensor(x_truth_log, norm_stats)
    truth_frames.append(frame_tensor(truth_solver_log.fields()["x"].detach().cpu()))
    truth_frames_norm.append(x_truth_log_norm.detach().cpu())

In [ ]:
path_truth_full = torch.stack(truth_frames, dim=0)
path_truth_full_norm = torch.stack(truth_frames_norm, dim=0)
# path_forecast_ens = torch.stack([torch.stack(h, dim=0) for h in hist_forecast], dim=0)

In [ ]:
from scipy.io import savemat
path_truth_np = path_truth_full.detach().cpu().numpy()
path_truth_norm_np = path_truth_full_norm.detach().cpu().numpy()

In [ ]:
path_pf_analysis_true = torch.stack(path_pf_analysis_true, dim=1)   # NPF x (N_DA_CYCLES + 1) x 1 x 3 x 1; index 0 is pre-DA

# Remove singleton dimensions: NPF x (N_DA_CYCLES + 1) x 3
path_pf_analysis_true_save = path_pf_analysis_true.squeeze(2).squeeze(-1).cpu().numpy()


In [ ]:
path_pf_analysis = torch.stack(path_pf_analysis, dim=1)   # NPF x (N_DA_CYCLES + 1) x 1 x 3 x 1; index 0 is pre-DA

# Remove singleton dimensions: NPF x (N_DA_CYCLES + 1) x 3
path_pf_analysis_save = path_pf_analysis.squeeze(2).squeeze(-1).cpu().numpy()

# Alignment diagnostic: both PF methods should start from exactly the same pre-DA ensemble.
print("Max |PF+true init - PF+AI init| =", float(torch.max(torch.abs(path_pf_analysis_true[:, 0] - path_pf_analysis[:, 0]))))


In [ ]:
rmse_pf_true = torch.tensor(rmse_truth_vs_pf_true, dtype = torch.float32)
spread_pf_true = torch.tensor(spread_pf_true, dtype = torch.float32)

In [ ]:
rmse_pf = torch.tensor(rmse_truth_vs_pf, dtype = torch.float32)
spread_pf= torch.tensor(spread_pf, dtype = torch.float32)

In [ ]:
rmse_forecast_np = rmse_pf.detach().cpu().numpy()
spread_forecast_np = spread_pf.detach().cpu().numpy()

In [ ]:
import os

In [ ]:
s_churn = 0.0

In [ ]:
# Free-run baseline from the same attractor-based initialization rule.
NFREE = NE

if NFREE > init_state_indices.numel():
    raise ValueError("NFREE exceeds the number of available attractor states.")

hist_free = [
    make_attractor_initialized_history(path_burn, init_state_indices[k], HISTORY)
    for k in range(NFREE)
]

In [ ]:
rnd_free = StackedRandomGenerator(device, [SEED_DM + k for k in range(NFREE)])

In [ ]:
core_kw = params.to_core_kwargs()
truth_free = Lorenz63Solver(
    path_burn[-1][0, :, 0].to(device=device, dtype=dtype), **core_kw
)
truth_free.set_state(state_after_burn)

In [ ]:
path_free_run = []
rmse_truth_vs_free = []
spread_free = []
free_cycle_times = []

In [ ]:
pbar_free = tqdm(range(N_DA_CYCLES), desc="Free-run cycles", unit="cycle")

for cycle in pbar_free:
    t_cycle_start = time.perf_counter()

    # Advance the same deterministic truth trajectory used by the DA methods.
    truth_free.step(timesteps_per_output)
    x_truth_free = frame_tensor(truth_free.fields()["x"].detach().cpu())

    cond_free = build_condition_from_history_batch(
        hist_free, HISTORY, norm_stats).to(device)

    if not torch.isfinite(cond_free).all():
        print(f"Stopping at cycle {cycle}: non-finite free-run conditions")
        break

    # Use the same latent distribution and seed sequence as the proposed DA run.
    lat_free = rnd_free.randn([NFREE, n_target, STATE_DIM, 1], device=device)
    # lat_free = LATENT_INFLATION * lat_free

    with torch.no_grad():
        sample_free_norm = edm_sampler_obs(
            net, lat_free, cond_free, obs=None,
            num_steps=NUM_STEPS, S_churn=s_churn
        ).to(torch.float32)

    if not torch.isfinite(sample_free_norm).all():
        print(f"Stopping at cycle {cycle}: non-finite free-run sample")
        break

    next_free = denormalize_tensor(sample_free_norm.cpu(), norm_stats)

    if not torch.isfinite(next_free).all():
        print(f"Stopping at cycle {cycle}: non-finite free-run physical state")
        break

    for k in range(NFREE):
        hist_free[k].append(next_free[k])

    path_free_run.append(next_free.clone())

    rmse_free, spr_free = ensemble_mean_rmse(hist_free, x_truth_free)

    if not all(np.isfinite([rmse_free, spr_free])):
        print(f"Stopping at cycle {cycle}: non-finite free-run RMSE/spread")
        break

    rmse_truth_vs_free.append(rmse_free)
    spread_free.append(spr_free)

    free_cycle_times.append(time.perf_counter() - t_cycle_start)
    avg_step = sum(free_cycle_times) / len(free_cycle_times)
    remaining = avg_step * (N_DA_CYCLES - cycle - 1)

    pbar_free.set_postfix(
        rmse=f"{rmse_free:.4f}",
        spr=f"{spr_free:.4f}",
        step=f"{avg_step:.2f}s",
        remaining=f"{remaining:.0f}s",
    )

n_free_completed = len(path_free_run)
print(f"Completed {n_free_completed}/{N_DA_CYCLES} free-run cycles.")

path_free_run = torch.stack(path_free_run, dim=1)  # NFREE x Nt x 1 x 3 x 1
path_free_run_save = path_free_run.squeeze(2).squeeze(-1).cpu().numpy()
rmse_free_save = np.asarray(rmse_truth_vs_free, dtype=np.float32)
spread_free_save = np.asarray(spread_free, dtype=np.float32)

In [ ]:
# --- Saving configuration: PF baselines and free run ---
# Choose your own output folder and file name before running this cell.
folder1 = "../../output_Lorenz63/WithTrueL63/Test_ScaledFactor"
filename_save = "Lorenz63_PF_and_FreeRun.mat"

os.makedirs(folder1, exist_ok=True)
save_path = os.path.join(folder1, filename_save)
print("PF/free-run output:", save_path)


In [ ]:
# Save both particle-filter baselines, the free run, and the common truth trajectory.
savemat(save_path, {
    "path_pf_analysis_true": path_pf_analysis_true_save,
    "path_pf_analysis_diffusion": path_pf_analysis_save,
    "path_free_run": path_free_run_save,
    "path_truth": path_truth_np,
    "rmse_pf_true": rmse_pf_true.detach().cpu().numpy(),
    "spread_pf_true": spread_pf_true.detach().cpu().numpy(),
    "rmse_pf_diffusion": rmse_pf.detach().cpu().numpy(),
    "spread_pf_diffusion": spread_pf.detach().cpu().numpy(),
    "ess_pf_diffusion": np.asarray(ess_hist),
    "rmse_free_run": rmse_free_save,
    "spread_free_run": spread_free_save,
    "obs_sigma": float(obs_sigma),
    "NPF": int(NPF),
    "NFREE": int(NFREE),
    "t_burn_in": int(T_BURN_IN),
    "Ncycle": int(N_DA_CYCLES),
})

print("Saved PF/free-run results to:", save_path)


## Phase 4B: Proposed DA method — nested tuning of \(\Lambda_L\) and \(\alpha_F\)

The tuning stage evaluates the Cartesian product of candidate diagonal factors
\[
\Lambda_L=\operatorname{diag}(\lambda_x,\lambda_y,\lambda_z),
\qquad
\alpha_F=\operatorname{diag}(\alpha_x,\alpha_y,\alpha_z).
\]

`LAMBDA_L_SETS` controls the componentwise likelihood-guidance strength, while
`ALPHA_F_SETS` controls the componentwise contribution of the weighted
ensemble-Jacobian term.

Every parameter pair is run from the same initial ensemble, truth state,
observations, observation-noise seeds, and diffusion latent seeds. Results are
collected silently and summarized after the sweep.

By default, the completed parameter pair with the smallest final RMSE is
selected. Set `BEST_PARAM_OVERRIDE` to one tested pair if you want to choose a
specific case manually. The selected pair is then rerun from scratch with its
full analysis history retained for saving and diagnostics.


In [ ]:
N_DA_CYCLES = 40

In [ ]:
NE = 800

In [ ]:
# Weighted Fbar-centered full 3x3 J_F regression.
# Lambda_L controls the overall component-wise likelihood guidance.
# alpha_F controls the component-wise contribution of the estimated J_F term.

NUM_STEPS = 60

BASE_LAMBDA_L_SET = (0.9825, 0.9055, 0.6225)
# BASE_ALPHA_F_SET = (0.8175, 0.7635, 1.005)

BASE_ALPHA_F_SET = (0.8075, 0.7535, 1.025)
# Edit or extend these lists directly if you want a wider/finer search.
# Edit or extend the candidate lists to widen or refine the search.
## 10000
LAMBDA_L_SETS = [
    (0.9450, 0.8680, 0.6225),
    (0.9425, 0.8655, 0.6225),
    (0.9400, 0.8630, 0.6225),
    (0.9375, 0.8605, 0.6225),
    (0.9350, 0.8580, 0.6225),
    (0.9325, 0.8555, 0.6225),
    (0.9300, 0.8530, 0.6225),
    (0.9275, 0.8505, 0.6225),
    (0.9250, 0.8480, 0.6225),
    (0.9225, 0.8455, 0.6225),
    (0.9200, 0.8430, 0.6225),
    (0.9175, 0.8405, 0.6225),
    (0.9150, 0.8380, 0.6225),
    (0.9125, 0.8355, 0.6225),
    (0.9100, 0.8330, 0.6225),
    (0.9075, 0.8305, 0.6225),
    (0.9050, 0.8280, 0.6225),
    (0.9025, 0.8255, 0.6225),
    (0.9000, 0.8230, 0.6225),
    (0.8975, 0.8205, 0.6225),
    (0.8950, 0.8180, 0.6225),
    (0.8925, 0.8155, 0.6225),
    (0.8900, 0.8130, 0.6225),
    (0.8875, 0.8105, 0.6225),
    (0.8850, 0.8080, 0.6225),
    (0.8825, 0.8055, 0.6225),
    (0.8800, 0.8030, 0.6225),
    (0.8775, 0.8005, 0.6225),
    (0.8750, 0.7980, 0.6225),
    (0.8725, 0.7955, 0.6225),
    (0.8700, 0.7930, 0.6225),
]

### 10100
# LAMBDA_L_SETS = [
#     (0.9440, 0.8670, 0.6225),
#     (0.9425, 0.8655, 0.6225),
#     (0.9410, 0.8640, 0.6225),
#     (0.9395, 0.8625, 0.6225),
#     (0.9380, 0.8610, 0.6225),
#     (0.9365, 0.8595, 0.6225),
#     (0.9350, 0.8580, 0.6225),
#     (0.9335, 0.8565, 0.6225),
#     (0.9320, 0.8550, 0.6225),
#     (0.9305, 0.8535, 0.6225),
#     (0.9290, 0.8520, 0.6225),
#     (0.9275, 0.8505, 0.6225),
#     (0.9260, 0.8490, 0.6225),
#     (0.9245, 0.8475, 0.6225),
#     (0.9230, 0.8460, 0.6225),
#     (0.9215, 0.8445, 0.6225),
#     (0.9200, 0.8430, 0.6225),
#     (0.9185, 0.8415, 0.6225),
#     (0.9170, 0.8400, 0.6225),
#     (0.9155, 0.8385, 0.6225),
#     (0.9140, 0.8370, 0.6225),
#     (0.9125, 0.8355, 0.6225),
#     (0.9110, 0.8340, 0.6225),
#     (0.9095, 0.8325, 0.6225),
#     (0.9080, 0.8310, 0.6225),
#     (0.9065, 0.8295, 0.6225),
#     (0.9050, 0.8280, 0.6225),
#     (0.9035, 0.8265, 0.6225),
#     (0.9020, 0.8250, 0.6225),
#     (0.9005, 0.8235, 0.6225),
#     (0.8990, 0.8220, 0.6225),
#     (0.8975, 0.8205, 0.6225),
#     (0.8960, 0.8190, 0.6225),
#     (0.8945, 0.8175, 0.6225),
#     (0.8930, 0.8160, 0.6225),
#     (0.8915, 0.8145, 0.6225),
#     (0.8900, 0.8130, 0.6225),
#     (0.8885, 0.8115, 0.6225),
#     (0.8870, 0.8100, 0.6225),
#     (0.8855, 0.8085, 0.6225),
#     (0.8840, 0.8070, 0.6225),
#     (0.8825, 0.8055, 0.6225),
#     (0.8810, 0.8040, 0.6225),
#     (0.8795, 0.8025, 0.6225),
#     (0.8780, 0.8010, 0.6225),
#     (0.8765, 0.7995, 0.6225),
#     (0.8750, 0.7980, 0.6225),
#     (0.8735, 0.7965, 0.6225),
#     (0.8720, 0.7950, 0.6225),
#     (0.8705, 0.7935, 0.6225),
#     (0.8690, 0.7920, 0.6225),
#     (0.8675, 0.7905, 0.6225),
#     (0.8660, 0.7890, 0.6225),
#     (0.8645, 0.7875, 0.6225),
#     (0.8630, 0.7860, 0.6225),
#     (0.8615, 0.7845, 0.6225),
#     (0.8600, 0.7830, 0.6225),
#     (0.8585, 0.7815, 0.6225),
#     (0.8570, 0.7800, 0.6225),
#     (0.8555, 0.7785, 0.6225),
#     (0.8540, 0.7770, 0.6225),
#     ]
ALPHA_F_SETS = [
    BASE_ALPHA_F_SET
]

INTERMEDIATE_GUIDANCE_MULTIPLIER_X = 1.0
INTERMEDIATE_GUIDANCE_MULTIPLIER_Y = 1.0
INTERMEDIATE_GUIDANCE_MULTIPLIER_Z = 6.0 ### 10000
# INTERMEDIATE_GUIDANCE_MULTIPLIER_Z = 1.0 ### 10100

# Weighted Fbar-centered J_F regression.
# h_t = JF_WEIGHT_BANDWIDTH_FACTOR * median_e ||U_e - Ubar||_2.
JF_WEIGHT_BANDWIDTH_FACTOR = 1.0
JF_RIDGE_REL = 1e-6
JF_RIDGE_ABS = 1e-10

RESIDUAL_SOFTEN_ALPHA = 1.0

# Fallback only if the loaded EDM network does not expose net.sigma_data.
EDM_SIGMA_DATA_FALLBACK = 0.5

# Used only by the optional observation-denominator correction.
SIGMA_DATA = 4
USE_OBS_DENOM = False


In [ ]:
LATENT_INFLATION = 1.0

In [ ]:
POSTERIOR_GAMMA_XY = 1.0
POSTERIOR_GAMMA_Z  = 1.0

In [ ]:
s_churn = 0

In [ ]:
K_CONTRACT = 0.5

In [ ]:
obs_flat_idx = make_random_l63_obs_idx(
        obs_count=obs_count,
        state_dim=STATE_DIM,
        seed=SEED + 100000+1,
        device=device,
    )

In [ ]:
# ================================================================
# PART 1: nested Lambda_L x alpha_F sweep
# ================================================================
# Every candidate pair uses exactly the same initial histories, truth,
# observations, observation-noise seeds, and diffusion latent seeds.

def run_da_for_parameters(lambda_l_set, alpha_f_set, keep_history=False, show_progress=False, desc=None):
    lambda_x, lambda_y, lambda_z = map(float, lambda_l_set)
    alpha_x, alpha_y, alpha_z = map(float, alpha_f_set)

    if desc is None:
        desc = f"Lambda_L={tuple(lambda_l_set)}, alpha_F={tuple(alpha_f_set)}"

    rnd_local = StackedRandomGenerator(device, [SEED_DM + k for k in range(NE)])

    hist_local = []
    if NE > init_state_indices.numel():
        raise ValueError("NE exceeds the number of available attractor states.")
    for k in range(NE):
        hist_local.append(make_attractor_initialized_history(path_burn, init_state_indices[k], HISTORY))

    rmse_local = []
    spread_local = []
    obs_rmse_local = []
    obs_values_local = []
    nll_last_local = []

    core_kw_local = params.to_core_kwargs()
    truth_local = Lorenz63Solver(path_burn[-1][0, :, 0].to(device=device, dtype=dtype), **core_kw_local)
    truth_local.set_state(state_after_burn)

    cycle_times_local = []
    pbar = tqdm(range(N_DA_CYCLES), desc=desc, unit="cycle", disable=not show_progress)

    for cycle in pbar:
        t_cycle_start = time.perf_counter()

        truth_local.step(timesteps_per_output)
        x_truth = frame_tensor(truth_local.fields()["x"].detach().cpu())
        x_truth_norm = normalize_tensor(x_truth, norm_stats)

        obs_noise_gen = torch.Generator(device=x_truth_norm.device)
        obs_noise_gen.manual_seed(SEED + 150000 + cycle)
        y_obs = observe_state(x_truth_norm, obs_flat_idx, obs_sigma, generator=obs_noise_gen).to(device)

        obs_values_local.append(y_obs.detach().cpu().reshape(-1))
        if not torch.isfinite(y_obs).all():
            break

        obs_rmse_local.append(observation_rmse(x_truth_norm, y_obs, obs_flat_idx))

        cond = build_condition_from_history_batch(hist_local, HISTORY, norm_stats).to(device)
        if not torch.isfinite(cond).all():
            break

        obs_cfg = {
            "y_obs": y_obs,
            "flat_idx": obs_flat_idx,
            "obs_sigma": obs_sigma,
            "norm_stats": norm_stats,
            "residual_soften_alpha": RESIDUAL_SOFTEN_ALPHA,
            "jf_weight_bandwidth_factor": JF_WEIGHT_BANDWIDTH_FACTOR,
            "jf_ridge_rel": JF_RIDGE_REL,
            "jf_ridge_abs": JF_RIDGE_ABS,
            "alpha_f_x": alpha_x,
            "alpha_f_y": alpha_y,
            "alpha_f_z": alpha_z,
            "likelihood_scale_x": lambda_x,
            "likelihood_scale_y": lambda_y,
            "likelihood_scale_z": lambda_z,
            "intermediate_guidance_multiplier_x": INTERMEDIATE_GUIDANCE_MULTIPLIER_X,
            "intermediate_guidance_multiplier_y": INTERMEDIATE_GUIDANCE_MULTIPLIER_Y,
            "intermediate_guidance_multiplier_z": INTERMEDIATE_GUIDANCE_MULTIPLIER_Z,
            "edm_sigma_data_fallback": EDM_SIGMA_DATA_FALLBACK,
            "sigma_data": SIGMA_DATA,
            "use_denom": USE_OBS_DENOM,
        }

        lat = LATENT_INFLATION * rnd_local.randn([NE, n_target, STATE_DIM, 1], device=device)
        sample_norm, nll_hist = edm_sampler_obs(net, lat, cond, obs=obs_cfg, num_steps=NUM_STEPS, S_churn=s_churn)

        if cycle > K_CONTRACT:
            ens_mean_norm = sample_norm.mean(dim=0, keepdim=True)
            gamma_vec = torch.tensor([POSTERIOR_GAMMA_XY, POSTERIOR_GAMMA_XY, POSTERIOR_GAMMA_Z], dtype=sample_norm.dtype, device=sample_norm.device).view(1, 1, STATE_DIM, 1)
            sample_norm = ens_mean_norm + gamma_vec * (sample_norm - ens_mean_norm)

        if not torch.isfinite(sample_norm).all():
            break

        next_analysis = denormalize_tensor(sample_norm.cpu().float(), norm_stats)
        if not torch.isfinite(next_analysis).all():
            break

        for k in range(NE):
            hist_local[k].append(next_analysis[k])

        rmse_a, spr_a = ensemble_mean_rmse(hist_local, x_truth)
        if not all(np.isfinite([rmse_a, spr_a])):
            for k in range(NE):
                hist_local[k].pop()
            break

        rmse_local.append(rmse_a)
        spread_local.append(spr_a)
        nll_last_local.append(nll_hist[-1] if nll_hist else float("nan"))

        cycle_times_local.append(time.perf_counter() - t_cycle_start)
        if show_progress:
            avg_step = sum(cycle_times_local) / len(cycle_times_local)
            remaining = avg_step * (N_DA_CYCLES - cycle - 1)
            pbar.set_postfix(rmse=f"{rmse_a:.4f}", step=f"{avg_step:.2f}s", remaining=f"{remaining:.0f}s")

    n_completed = len(rmse_local)
    final_rmse = float(rmse_local[-1]) if rmse_local else float("nan")
    mean_rmse = float(np.mean(rmse_local)) if rmse_local else float("nan")

    result = {
        "lambda_l_set": tuple(map(float, lambda_l_set)),
        "alpha_f_set": tuple(map(float, alpha_f_set)),
        "rmse_analysis": np.asarray(rmse_local, dtype=np.float64),
        "spread_analysis": np.asarray(spread_local, dtype=np.float64),
        "obs_rmse": np.asarray(obs_rmse_local, dtype=np.float64),
        "obs_values": obs_values_local,
        "neg_log_likelihood_last": np.asarray(nll_last_local, dtype=np.float64),
        "n_completed": n_completed,
        "final_rmse": final_rmse,
        "mean_rmse": mean_rmse,
    }

    if keep_history:
        result["hist_analysis"] = hist_local

    return result


all_sweep_results = {}
n_parameter_runs = len(LAMBDA_L_SETS) * len(ALPHA_F_SETS)
sweep_progress = tqdm(total=n_parameter_runs, desc="Parameter sweep", unit="run")

for lambda_l_set in LAMBDA_L_SETS:
    for alpha_f_set in ALPHA_F_SETS:
        key = (tuple(map(float, lambda_l_set)), tuple(map(float, alpha_f_set)))
        all_sweep_results[key] = run_da_for_parameters(lambda_l_set, alpha_f_set, keep_history=False, show_progress=False)
        sweep_progress.update(1)

sweep_progress.close()


In [ ]:
# ================================================================
# PART 1b: show the completed sweep once and select the best pair
# ================================================================
import pandas as pd
from IPython.display import display

# By default, "best" means the smallest FINAL RMSE among runs that completed
# every DA cycle. Set BEST_PARAM_OVERRIDE=((lambda_x,lambda_y,lambda_z),
# (alpha_x,alpha_y,alpha_z)) to choose a pair manually.
BEST_PARAM_OVERRIDE = None
BEST_SELECTION_METRIC = "final_rmse"

sweep_rows = []
for (lambda_l_set, alpha_f_set), result in all_sweep_results.items():
    sweep_rows.append({
        "lambda_x": lambda_l_set[0],
        "lambda_y": lambda_l_set[1],
        "lambda_z": lambda_l_set[2],
        "alpha_x": alpha_f_set[0],
        "alpha_y": alpha_f_set[1],
        "alpha_z": alpha_f_set[2],
        "completed": result["n_completed"],
        "final_RMSE": result["final_rmse"],
        "mean_RMSE": result["mean_rmse"],
    })

sweep_table = pd.DataFrame(sweep_rows)
sweep_table["completed_all"] = sweep_table["completed"] == N_DA_CYCLES
sweep_table = sweep_table.sort_values(["completed_all", "final_RMSE", "mean_RMSE"], ascending=[False, True, True]).reset_index(drop=True)
display(sweep_table)

if BEST_PARAM_OVERRIDE is not None:
    BEST_LAMBDA_L_SET = tuple(map(float, BEST_PARAM_OVERRIDE[0]))
    BEST_ALPHA_F_SET = tuple(map(float, BEST_PARAM_OVERRIDE[1]))
    best_key = (BEST_LAMBDA_L_SET, BEST_ALPHA_F_SET)
    if best_key not in all_sweep_results:
        raise ValueError("BEST_PARAM_OVERRIDE must contain one Lambda_L set from LAMBDA_L_SETS and one alpha_F set from ALPHA_F_SETS.")
else:
    eligible = {key: result for key, result in all_sweep_results.items() if result["n_completed"] == N_DA_CYCLES and np.isfinite(result[BEST_SELECTION_METRIC])}
    if not eligible:
        raise RuntimeError("No parameter pair completed all DA cycles; inspect the sweep table before selecting a best pair.")
    best_key = min(eligible, key=lambda key: eligible[key][BEST_SELECTION_METRIC])
    BEST_LAMBDA_L_SET, BEST_ALPHA_F_SET = best_key

print(f"Selected Lambda_L = {BEST_LAMBDA_L_SET}")
print(f"Selected alpha_F = {BEST_ALPHA_F_SET}")
print(f"Selection metric = {BEST_SELECTION_METRIC}: {all_sweep_results[best_key][BEST_SELECTION_METRIC]:.6f}")


In [ ]:
# ================================================================
# PART 2: rerun the selected Lambda_L / alpha_F pair and keep the results
# ================================================================
LIKELIHOOD_SCALE_X, LIKELIHOOD_SCALE_Y, LIKELIHOOD_SCALE_Z = BEST_LAMBDA_L_SET
ALPHA_F_X, ALPHA_F_Y, ALPHA_F_Z = BEST_ALPHA_F_SET

best_da_result = run_da_for_parameters(BEST_LAMBDA_L_SET, BEST_ALPHA_F_SET, keep_history=True, show_progress=True, desc=f"Selected Lambda_L={BEST_LAMBDA_L_SET}, alpha_F={BEST_ALPHA_F_SET}")

if best_da_result["n_completed"] != N_DA_CYCLES:
    raise RuntimeError("The selected rerun did not complete all DA cycles.")

hist_analysis = best_da_result["hist_analysis"]
rmse_truth_vs_analysis = best_da_result["rmse_analysis"].tolist()
spread_analysis = best_da_result["spread_analysis"].tolist()
obs_rmse = best_da_result["obs_rmse"].tolist()
obs_values = best_da_result["obs_values"]
neg_log_likelihood_last = best_da_result["neg_log_likelihood_last"].tolist()

path_analysis_ens = torch.stack([torch.stack(h, dim=0) for h in hist_analysis], dim=0)

print(f"Best-pair rerun complete: Lambda_L={BEST_LAMBDA_L_SET}, alpha_F={BEST_ALPHA_F_SET}, final RMSE={rmse_truth_vs_analysis[-1]:.6f}")


In [ ]:
path_analysis_np = path_analysis_ens.detach().cpu().numpy()
spread_analysis_np = torch.tensor(spread_analysis, dtype = torch.float32).detach().cpu().numpy()

In [ ]:
rmse_analysis_np = torch.tensor(rmse_truth_vs_analysis, dtype =torch.float32).detach().cpu().numpy()
# rmse_obs_analysis_np = torch.tensor(rmse_obs_analysis, dtype =torch.float32).detach().cpu().numpy()
# rmse_unobs_analysis_np = torch.tensor(rmse_unobs_analysis, dtype =torch.float32).detach().cpu().numpy()

In [ ]:
path_analysis = path_analysis_ens.mean(dim=0)

## Saving the proposed-method results

Choose the output folder and MATLAB file name below. The shared notebook does
not assume a fixed project-specific save location or automatically construct a
long file name.


In [ ]:
# --- Saving configuration: proposed DA method ---
folder1 = "../../output_Lorenz63/WithTrueL63/Test_ScaledFactor"
filename_save = "Lorenz63_WeightedFbarJF_AdaptiveDT.mat"

os.makedirs(folder1, exist_ok=True)
save_path = os.path.join(folder1, filename_save)
print("Proposed-method output:", save_path)


In [ ]:
from scipy.io import savemat

sweep_lambda_l = np.asarray([[row["lambda_x"], row["lambda_y"], row["lambda_z"]] for row in sweep_rows], dtype=np.float64)
sweep_alpha_f = np.asarray([[row["alpha_x"], row["alpha_y"], row["alpha_z"]] for row in sweep_rows], dtype=np.float64)
sweep_completed = np.asarray([row["completed"] for row in sweep_rows], dtype=np.int64)
sweep_final_rmse = np.asarray([row["final_RMSE"] for row in sweep_rows], dtype=np.float64)
sweep_mean_rmse = np.asarray([row["mean_RMSE"] for row in sweep_rows], dtype=np.float64)

savemat(save_path, {
    "path_analysis": path_analysis_np,
    "path_truth": path_truth_np,
    "rmse_analysis": rmse_analysis_np,
    "spread_analysis": spread_analysis_np,
    "obs_count": int(obs_count),
    "NE": int(NE),
    "Ncycle": int(N_DA_CYCLES),
    "t_burn_in": int(T_BURN_IN),
    "obs_rmse_all": np.asarray(obs_rmse, dtype=np.float64),
    "lambda_l_x": float(LIKELIHOOD_SCALE_X),
    "lambda_l_y": float(LIKELIHOOD_SCALE_Y),
    "lambda_l_z": float(LIKELIHOOD_SCALE_Z),
    "alpha_f_x": float(ALPHA_F_X),
    "alpha_f_y": float(ALPHA_F_Y),
    "alpha_f_z": float(ALPHA_F_Z),
    "jf_weight_bandwidth_factor": float(JF_WEIGHT_BANDWIDTH_FACTOR),
    "jf_ridge_rel": float(JF_RIDGE_REL),
    "jf_ridge_abs": float(JF_RIDGE_ABS),
    "sweep_lambda_l": sweep_lambda_l,
    "sweep_alpha_f": sweep_alpha_f,
    "sweep_completed": sweep_completed,
    "sweep_final_rmse": sweep_final_rmse,
    "sweep_mean_rmse": sweep_mean_rmse,
})

print("Saved proposed-method results to:", save_path)


## Phase 5: Quick Python diagnostic

The main visualization is intended to be produced in MATLAB from the saved
`.mat` files. This section provides only a compact sanity check.

For each assimilation cycle, the shaded region is the 5th–95th percentile range
of the **memberwise state RMSE** across the ensemble. The solid line is the RMSE
of the **ensemble-mean analysis**. The shaded region is an ensemble percentile
band, not a frequentist confidence interval.


In [ ]:
n_done = len(rmse_truth_vs_analysis)

if n_done == 0:
    print("No completed DA cycles to plot.")
else:
    truth_diag = Lorenz63Solver(path_burn[-1][0, :, 0].to(device=device, dtype=dtype), **params.to_core_kwargs())
    truth_diag.set_state(state_after_burn)
    truth_cycle = []

    for _ in range(n_done):
        truth_diag.step(timesteps_per_output)
        truth_cycle.append(truth_diag.fields()["x"].detach().cpu().numpy().reshape(STATE_DIM))

    truth_cycle = np.asarray(truth_cycle, dtype=np.float64)
    analysis_cycle = path_analysis_np[:, -n_done:, 0, :, 0].astype(np.float64)

    member_rmse = np.sqrt(np.mean((analysis_cycle - truth_cycle[None, :, :]) ** 2, axis=2))
    rmse_low = np.percentile(member_rmse, 5, axis=0)
    rmse_high = np.percentile(member_rmse, 95, axis=0)
    rmse_median = np.median(member_rmse, axis=0)
    rmse_ens_mean = np.asarray(rmse_truth_vs_analysis, dtype=np.float64)

    cycles = np.arange(1, n_done + 1)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.fill_between(cycles, rmse_low, rmse_high, alpha=0.20, label="member RMSE: 5th–95th percentile")
    ax.plot(cycles, rmse_median, "--", linewidth=1.2, label="median member RMSE")
    ax.plot(cycles, rmse_ens_mean, linewidth=2.0, label="ensemble-mean RMSE")
    ax.set_xlabel("assimilation cycle")
    ax.set_ylabel("RMSE")
    ax.set_title("Analysis RMSE diagnostic")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("Quick Python diagnostic only; use the saved MATLAB files for the main visualization.")
